# 01 – E-commerce Project Configuration

Central configuration imported by Bronze, Silver, SCD, Gold, and data-quality notebooks.

## Confirmed raw folder convention

The current ADLS example is:

```text
raw/
└── category_translation/
    └── 2026/
        └── 10/
            └── 31/
                └── category_translation.parquet
```

Therefore, source data is **Parquet** and the folder structure is:

`raw/<entity>/<yyyy>/<MM>/<dd>/<filename>.parquet`

Use the date-aware path helpers below. Do not assume every source is CSV or that all source entities use the same file format; configure each entity explicitly.


## 1. Environment and Unity Catalog

In [0]:
ENVIRONMENT = "dev"
PROJECT_NAME = "azure_ecom_etl"

UC_CATALOG = "adb_ecom_dev_01"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

BRONZE_DB = f"{UC_CATALOG}.{BRONZE_SCHEMA}"
SILVER_DB = f"{UC_CATALOG}.{SILVER_SCHEMA}"
GOLD_DB = f"{UC_CATALOG}.{GOLD_SCHEMA}"


## 2. ADLS roots and curated layers

In [0]:
STORAGE_ACCOUNT = "stecraw001"
RAW_CONTAINER = "raw"
CURATED_CONTAINER = "curated"

RAW_ROOT = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
CURATED_ROOT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

BRONZE_ROOT = f"{CURATED_ROOT}bronze/"
SILVER_ROOT = f"{CURATED_ROOT}silver/"
GOLD_ROOT = f"{CURATED_ROOT}gold/"


## 3. Raw source entities and formats

The entity list below matches the folders currently visible in the ADLS `raw` container. ADF landing folders follow `raw/<entity>/YYYY/MM/DD/`. The shared screenshot confirms `category_translation` is Parquet. Other formats remain unset until their actual ADF output files are checked; do not assume every source uses the same format.

In [0]:
# Exact entity folder names currently present in the ADLS raw container.
# Expected ADF landing layout: raw/<entity>/YYYY/MM/DD/<filename>
RAW_ENTITIES = [
    "category_translation",
    "customer",
    "geolocation",
    "jsontest",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "product_category_name_translation",
    "products",
    "sellers",
]

# Build one root path per entity using the exact storage folder names.
RAW_ENTITY_PATHS = {
    entity: f"{RAW_ROOT}{entity}/"
    for entity in RAW_ENTITIES
}
# print(RAW_ENTITY_PATHS)
# Only category_translation has been confirmed as Parquet from the shared screenshot.
# Set the remaining formats after checking their actual ADF output files.
# All source entities are stored in Parquet format

RAW_ENTITY_FORMATS = {
    entity: "parquet"
    for entity in RAW_ENTITIES
}
# print (RAW_ENTITY_FORMATS)


## 4. Date-aware raw path helpers

These helpers build paths such as `raw/category_translation/2026/10/31/` and optionally append the filename.

In [0]:
def raw_date_path(entity: str, year: int, month: int, day: int) -> str:
    """Return the ADF landing folder: raw/<entity>/YYYY/MM/DD/."""
    if entity not in RAW_ENTITY_PATHS:
        raise ValueError(f"Unknown raw entity: {entity}. Add it to RAW_ENTITY_PATHS first.")
    if not (1 <= month <= 12 and 1 <= day <= 31):
        raise ValueError("Month must be 1-12 and day must be 1-31.")
    return f"{RAW_ENTITY_PATHS[entity]}{year:04d}/{month:02d}/{day:02d}/"


def raw_file_path(entity: str, year: int, month: int, day: int,
                  filename: str | None = None) -> str:
    """Return a dated landing folder, or a specific file within it."""
    folder = raw_date_path(entity, year, month, day)
    return folder if filename is None else folder + filename


def raw_format(entity: str) -> str:
    """Return the confirmed source format; fail clearly if it is not configured yet."""
    if entity not in RAW_ENTITY_FORMATS:
        raise ValueError(f"Unknown raw entity format: {entity}")
    fmt = RAW_ENTITY_FORMATS[entity]
    if not fmt:
        raise ValueError(
            f"Source format for '{entity}' is not confirmed. Check its ADF output, "
            "then set RAW_ENTITY_FORMATS[entity]."
        )
    return fmt


# Exact file path matching the ADLS screenshot.
RAW_CATEGORY_TRANSLATION_20261031 = raw_file_path(
    "category_translation", 2026, 10, 31, "category_translation.parquet"
)

print("Category translation file:", RAW_CATEGORY_TRANSLATION_20261031)


## 5. Bronze external Delta locations

In [0]:
# Bronze is stored as external Delta under curated/bronze/<entity>/.
BRONZE_ENTITY_PATHS = {
    entity: f"{BRONZE_ROOT}{entity}/"
    for entity in RAW_ENTITIES
}

# Convenience example; use the matching entity key when processing other datasets.
BRONZE_CATEGORY_TRANSLATION_PATH = BRONZE_ENTITY_PATHS["category_translation"]



## 6. Shared processing and audit settings

In [0]:
DELTA_FORMAT = "delta"

PARQUET_READ_OPTIONS = {}
CSV_READ_OPTIONS = {
    "header": "true",
    "inferSchema": "true"
}

INGESTION_TIMESTAMP_COLUMN = "_ingested_at"
SOURCE_FILE_COLUMN = "_source_file"
BATCH_ID_COLUMN = "_batch_id"


## 7. Configuration validation and example read

In [0]:
assert ENVIRONMENT in {"dev", "test", "prod"}
assert UC_CATALOG and STORAGE_ACCOUNT
assert RAW_ROOT.startswith("abfss://raw@")
assert CURATED_ROOT.startswith("abfss://curated@")

print("Environment:", ENVIRONMENT)
print("Catalog:", UC_CATALOG)
print("Raw root:", RAW_ROOT)
print("Curated root:", CURATED_ROOT)
print("Example dated file:", RAW_CATEGORY_TRANSLATION_20261031)

# Optional read test for the exact file shown in your ADLS screenshot.
# Uncomment only after confirming that this exact file exists and permissions work.
#
# df_category_translation = (
#     spark.read
#          .format("parquet")
#          .load(RAW_CATEGORY_TRANSLATION_20261031)
# )
# display(df_category_translation.limit(10))


## 8. Import this configuration in another notebook

From a notebook inside `02_Bronze`:

```python
%run ../01_Configuration/01_Project_Config

source_path = raw_date_path("category_translation", 2026, 10, 31)
df = spark.read.format("parquet").load(source_path)
display(df)
```

For a date-driven pipeline, pass the ingestion year, month, and day as notebook/job parameters rather than hardcoding them. We will implement that in the Bronze ingestion notebook.
